# D² Pruning — CIFAR-10 trên Kaggle
Dùng dataset đã attach, không tải CIFAR-10. Bật GPU và Internet (chỉ để clone mã nguồn).

Notebook mặc định chạy full-data, Random và D² tại pruning 70%, một seed. Đổi PRUNING_RATES và SEEDS để mở rộng. Đây là reproduction của implementation công khai; hard cutoff chưa được xác nhận cho mọi mức pruning, nên kết quả phải được báo cáo kèm cấu hình.

Nguồn: https://arxiv.org/html/2310.07931v1 ; https://github.com/adymaharana/d2pruning


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, hashlib, re
import numpy as np
import pandas as pd
import torch
SOURCE = Path('/kaggle/input/datasets/vnnhhiu/cifar-10/cifar-10-batches-py')
ROOT = Path('/kaggle/working/d2-reproduction')
REPO = ROOT / 'repo'
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
COMMIT = 'bf1ea46b22cc79549df421fcd92c9cd2d3408008'
SEEDS = [0]  # dùng [0, 1, 2] cho đánh giá nhiều seed
PRUNING_RATES = [70]  # có thể dùng [30, 50, 70, 80, 90]
STEPS = 40000  # cùng ngân sách cho full-data và các tập con
# Cutoff thử nghiệm, KHÔNG phải giá trị đã xác nhận từ bài báo.
# Không áp dụng 0.4 của ví dụ README cho mọi pruning rate.
MIS_RATIO = {30: 0.0, 50: 0.0, 70: 0.0, 80: 0.0, 90: 0.0}
# Có thể gắn output baseline cũ làm input rồi điền thư mục chứa ckpt-last.pt và td-all-data.pickle.
EXISTING_BASELINE = None
assert torch.cuda.is_available(), 'Bật GPU trong Settings của Kaggle'
assert SOURCE.is_dir(), f'Không tìm thấy {SOURCE}'
for name in [f'data_batch_{i}' for i in range(1,6)] + ['test_batch', 'batches.meta']:
    assert (SOURCE / name).is_file(), name
for p in [ROOT, DATA / 'cifar10', RESULTS]: p.mkdir(parents=True, exist_ok=True)
link = DATA / 'cifar10' / 'cifar-10-batches-py'
if not link.exists(): link.symlink_to(SOURCE, target_is_directory=True)
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
def command(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)
if not REPO.exists():
    command(['git', 'clone', 'https://github.com/adymaharana/d2pruning.git', REPO])
command(['git', 'checkout', '--detach', COMMIT], cwd=REPO)
# Dùng thư viện sẵn có của Kaggle thay vì ép scikit-learn 0.24 trên Python mới.
import scipy, sklearn, torchvision, matplotlib, tqdm
from torchvision.datasets import CIFAR10
train_data = CIFAR10(str(DATA / 'cifar10'), train=True, download=False)
test_data = CIFAR10(str(DATA / 'cifar10'), train=False, download=False)
assert len(train_data)==50000 and len(test_data)==10000
print('Dataset verified:', len(train_data), len(test_data))


## Điều chỉnh tương thích và kiểm soát thí nghiệm
Chỉ sửa hạ tầng chạy: worker, đường dẫn, seed, chuyển kiểu indices khi vẽ, và số bước cập nhật chính xác. Giữ nguyên cách tính features/scores và thuật toán của repo. Cấm download trong loader; nếu dataset thiếu hoặc sai checksum, dừng ngay.


In [ ]:
def patch(file, old, new):
    p = REPO / file
    s = p.read_text()
    if old in s: p.write_text(s.replace(old, new))
    elif new not in s: raise RuntimeError(f'Không tìm thấy vị trí patch: {file}: {old}')
for file in ['train.py', 'generate_importance_score.py']:
    patch(file, 'num_workers=16', 'num_workers=0')
patch('core/data/MiscDataset.py', 'download=True', 'download=False')
patch('train.py', 'start_time = datetime.now()', 'start_time = time.time()')
patch('train.py', 'trainer.train(current_epoch, -1,', 'trainer.train(current_epoch, iterations_epoch,')
patch('train.py', 'num_of_iterations -= iterations_per_epoch', 'num_of_iterations -= iterations_epoch')
patch('selection_mp.py', 'coreset_index.numpy().astype(np.int32)', 'np.asarray(coreset_index).astype(np.int32)')
(REPO.parent / 'data-pruning-analysis').mkdir(exist_ok=True)
# Seed phải được đặt trong subprocess thực thi train.py.
(REPO / 'seeded_run.py').write_text("""import os, sys, random, runpy
import numpy as np
import torch
seed = int(os.environ['D2_SEED'])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
script = sys.argv.pop(1)
runpy.run_path(script, run_name='__main__')
""")
def run(script, args, seed):
    env = dict(os.environ, D2_SEED=str(seed), PYTHONHASHSEED=str(seed), OMP_NUM_THREADS='2')
    subprocess.run([sys.executable, 'seeded_run.py', script, *map(str,args)], cwd=REPO, env=env, check=True)
manifest = dict(commit=COMMIT, seeds=SEEDS, pruning_rates=PRUNING_RATES,
                steps=STEPS, mis_ratio=MIS_RATIO, cutoff_verified=False,
                torch=torch.__version__, torchvision=torchvision.__version__, sklearn=sklearn.__version__)
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2))


## Baseline và trích xuất đặc trưng
Nếu dùng baseline cũ, scores và features sẽ dùng chung cho các seed downstream; đây không phải các lần lặp độc lập toàn bộ pipeline. Baseline cũ 200 epoch có 39.200 bước, khác ngân sách 40.000 mặc định ở đây. Không ghi đè thư mục thí nghiệm đã có: nếu muốn chạy lại, chọn ROOT mới.


In [ ]:
import pickle
baselines = {}
for seed in SEEDS:
    base = RESULTS / f'seed-{seed}'
    folder = base / 'all-data'
    folder.mkdir(parents=True, exist_ok=True)
    if EXISTING_BASELINE:
        for name in ['ckpt-last.pt', 'td-all-data.pickle']:
            src = Path(EXISTING_BASELINE) / name
            assert src.is_file(), src
            if not (folder / name).exists(): shutil.copy2(src, folder / name)
    elif not (folder / 'complete.json').exists():
        assert not (folder / 'log-train-all-data.log').exists(), 'Run chưa hoàn tất: dùng ROOT mới để chạy lại'
        run('train.py', ['--dataset','cifar10','--data-dir',DATA,'--gpuid','0',
            '--iterations',STEPS,'--lr','0.1','--network','resnet18','--batch-size','256',
            '--task-name','all-data','--base-dir',base], seed)
        (folder / 'complete.json').write_text(json.dumps(manifest))
    score = folder / 'data-score-all-data.pickle'
    feature = folder / 'train-features-conv3-all-data.npy'
    if not (folder / 'features-complete.json').exists():
        run('generate_importance_score.py', ['--dataset','cifar10','--data-dir',DATA,
            '--gpuid','0','--base-dir',base,'--task-name','all-data','--feature'], seed)
        (folder / 'features-complete.json').write_text('{}')
    with score.open('rb') as f: scores = pickle.load(f)
    features = np.load(feature, mmap_mode='r')
    assert len(scores['forgetting']) == 50000 and features.shape[0] == 50000
    assert np.isfinite(features).all()
    baselines[seed] = (base, score, feature)
    print(seed, features.shape)


## D² và Random
Thông số k, gamma lấy từ Table 4. MIS_RATIO là lựa chọn thử nghiệm đang được ghi lại, cần xác nhận hard cutoff trước khi tuyên bố reproduce chính xác Table 1. Mỗi model tập con khởi tạo mới. Random cũng nhận data-score-path vì implementation hiện tại luôn đọc file scores.


In [ ]:
PARAMS = {30:(10,0.9), 50:(5,1.0), 70:(1,0.1), 80:(2,0.0), 90:(2,0.0)}
for seed, (base, score, feature) in baselines.items():
    for pruning in PRUNING_RATES:
        ratio = (100-pruning)/100
        k, gamma = PARAMS[pruning]
        assert 0 <= MIS_RATIO[pruning] < 1-ratio
        for method in ['random','d2']:
            task = f'{method}-prune-{pruning}'
            folder = base / task
            if (folder / 'complete.json').exists(): continue
            assert not folder.exists(), f'Run chưa hoàn tất: {folder}; dùng ROOT mới'
            args = ['--dataset','cifar10','--data-dir',DATA,'--gpuid','0',
                    '--iterations',STEPS,'--batch-size','256','--lr','0.1','--network','resnet18',
                    '--base-dir',base,'--task-name',task,'--coreset','--coreset-ratio',ratio,
                    '--data-score-path',score,'--coreset-key','forgetting','--label-balanced']
            if method == 'random': args += ['--coreset-mode','random']
            else: args += ['--coreset-mode','class','--budget-mode','uniform','--sampling-mode','graph',
                          '--feature-path',feature,'--mis-ratio',MIS_RATIO[pruning],
                          '--n-neighbor',k,'--gamma',gamma,'--stratas','25',
                          '--graph-mode','sum','--graph-sampling-mode','weighted']
            run('train.py', args, seed)
            idx = np.load(folder / f'coreset-{task}.npy').astype(int)
            assert len(idx) == round(50000*ratio) and len(np.unique(idx)) == len(idx)
            assert idx.min() >= 0 and idx.max() < 50000
            print('Class counts:', np.bincount(np.array(train_data.targets)[idx], minlength=10))
            (folder / 'complete.json').write_text(json.dumps(dict(seed=seed, method=method,
                pruning=pruning, steps=STEPS, mis_ratio=MIS_RATIO[pruning] if method=='d2' else None)))


In [ ]:
paper = {'random':{0:95.5,30:94.3,50:93.4,70:90.9,80:88.0,90:79.0},
         'd2':{30:95.7,50:94.9,70:93.3,80:91.4,90:87.1}}
rows=[]
for seed in SEEDS:
    base=RESULTS / f'seed-{seed}'
    tasks=[('random',0,'all-data')] + [(m,p,f'{m}-prune-{p}') for p in PRUNING_RATES for m in ['random','d2']]
    for method,pruning,task in tasks:
        log=base/task/f'log-train-{task}.log'
        if not log.exists(): continue
        text=log.read_text()
        best=re.findall(r'Best acc: ([0-9.]+)',text)
        last=re.findall(r'Test Accuracy: ([0-9.]+)',text)
        if not best or not last: continue
        best_pct=float(best[-1])*100  # dòng Best acc cuối là tỷ lệ 0..1
        rows.append(dict(seed=seed,method='full' if pruning==0 else method,pruning=pruning,
                         best_accuracy=best_pct,last_accuracy=float(last[-1]),
                         paper_accuracy=paper[method][pruning],delta_pp=best_pct-paper[method][pruning]))
results=pd.DataFrame(rows)
results.to_csv(RESULTS/'results.csv',index=False)
display(results)
if len(SEEDS)>1:
    summary=results.groupby(['method','pruning']).agg(mean=('best_accuracy','mean'),std=('best_accuracy','std'),n=('seed','count'))
    display(summary)
    summary.to_csv(RESULTS/'summary.csv')
print('Lưu toàn bộ output:', RESULTS)
print('Kết quả D2 dùng cutoff thử nghiệm; chênh lệch số học không tự chứng minh tương đương thống kê.')


## Cách báo cáo
Báo cáo cả best và last accuracy, số bước, seed, cutoff, commit và môi trường. Table 1 không cung cấp độ lệch chuẩn ở các ô này; không đặt ngưỡng thành công tùy ý. Để mở rộng, chạy các pruning rate còn lại, nhiều seed và thêm CCS. Notebook này chỉ bao phủ CIFAR-10, chưa tái lập toàn bộ bài báo.
